# SST Evaluation — VQAv2 Benchmark
**Second benchmark notebook for the SST vs. VLM paper.**

Key differences from `sst_eval_main.ipynb`:
- Dataset: **VQAv2** (COCO val2014) instead of GQA
- No oracle scene graphs — all SST variants use **detected SST** (YOLO + colour/size attributes)
- Accuracy: both **exact match** and **VQAv2 soft accuracy** (min(count/3, 1.0))
- Balanced sampling across VQAv2 answer types: `yes/no`, `number`, `other`
- Final cell: **cross-benchmark comparison** against GQA results

> Run `sst_eval_main.ipynb` first so GQA summary CSVs exist for the comparison cell.


In [2]:
import sys, os, json, re, time, random, math, base64
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# TODO: once stable, move shared helpers to src/utils.py and import from there
try:
    import tiktoken
    TOKEN_ENCODER = tiktoken.get_encoding("cl100k_base")
    print("tiktoken loaded — using subword token counts")
except Exception:
    TOKEN_ENCODER = None
    print("tiktoken not found — using whitespace token counts")

try:
    import ollama
    print("ollama package loaded")
except Exception:
    ollama = None
    print("ollama not found — run: pip install ollama")

try:
    from ultralytics import YOLO
    import cv2
    print("ultralytics + cv2 loaded")
except Exception:
    YOLO = None
    print("ultralytics/cv2 not found — SST building cells will fail")

try:
    import easyocr
    OCR_READER = easyocr.Reader(["en"], verbose=False)
    print("easyocr loaded")
except Exception:
    OCR_READER = None
    print("easyocr not found — text extraction disabled")

random.seed(42)
np.random.seed(42)


tiktoken not found — using whitespace token counts
ollama package loaded
ultralytics + cv2 loaded
easyocr loaded


In [3]:
# ── Paths ─────────────────────────────────────────────────────────────────
PROJECT_ROOT   = Path("..").resolve()
VQAV2_DIR      = PROJECT_ROOT / "data" / "vqav2"
COCO_IMAGE_DIR = PROJECT_ROOT / "data" / "coco_images_val2014"
RESULTS_DIR    = PROJECT_ROOT / "outputs" / "results"
GQA_RESULTS    = RESULTS_DIR   # where sst_eval_main.ipynb saved its CSVs

for d in [VQAV2_DIR, COCO_IMAGE_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Evaluation config ──────────────────────────────────────────────────────
# Balanced across answer_type: yes/no, number, other
SAMPLES_PER_TYPE = 400          # 400 × 3 = 1200 total questions
MAX_SAMPLES      = SAMPLES_PER_TYPE * 3
CHECKPOINT_FREQ  = 50
PREFIX           = f"vqav2_{MAX_SAMPLES}_sst_eval"
CHECKPOINT       = RESULTS_DIR / f"{PREFIX}_checkpoint.csv"

# VQAv2 question/annotation files (val2014 split)
QUESTIONS_FILE   = VQAV2_DIR / "v2_OpenEnded_mscoco_val2014_questions.json"
ANNOTATIONS_FILE = VQAV2_DIR / "v2_mscoco_val2014_annotations.json"

# LLaVA output file
LLAVA_OUT_V2     = RESULTS_DIR / f"{PREFIX}_llava_results.csv"

# LLaVA token reference (ViT-L/14 @ 336px = 576 visual tokens)
LLAVA_VISUAL_TOKENS = 576

print("Project root:", PROJECT_ROOT)
print("Results dir: ", RESULTS_DIR)
print(f"Target: {MAX_SAMPLES} samples ({SAMPLES_PER_TYPE} per answer type)")


Project root: /Users/alisha/Documents/CSE427_SST_Project
Results dir:  /Users/alisha/Documents/CSE427_SST_Project/outputs/results
Target: 1200 samples (400 per answer type)


## Data download

Download the following files and place them in `data/vqav2/`:

```bash
# Questions (val2014)
wget https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Val_mscoco.zip
unzip v2_Questions_Val_mscoco.zip -d data/vqav2/

# Annotations (val2014)
wget https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Val_mscoco.zip
unzip v2_Annotations_Val_mscoco.zip -d data/vqav2/

# COCO val2014 images (~6 GB) — only needed for detected SST building
wget http://images.cocodataset.org/zips/val2014.zip
unzip val2014.zip -d data/coco_images_val2014/
```

Expected filenames after unzip:
- `data/vqav2/v2_OpenEnded_mscoco_val2014_questions.json`
- `data/vqav2/v2_mscoco_val2014_annotations.json`
- `data/coco_images_val2014/COCO_val2014_000000XXXXXX.jpg`


In [ ]:
# ── Load VQAv2 questions + annotations ────────────────────────────────────
with open(QUESTIONS_FILE)   as f: q_data  = json.load(f)
with open(ANNOTATIONS_FILE) as f: ann_data = json.load(f)

q_df   = pd.DataFrame(q_data["questions"])     # question_id, image_id, question
ann_df = pd.DataFrame(ann_data["annotations"])  # question_id, answers, answer_type, question_type

merged = q_df.merge(ann_df[["question_id","answers","answer_type","question_type"]],
                    on="question_id", how="inner")

print(f"Total questions: {len(merged):,}")
print("Answer type distribution:")
print(merged["answer_type"].value_counts().to_string())


In [ ]:
# ── Balanced sampling across answer types ─────────────────────────────────
def get_majority_answer(answers: list) -> str:
    """Most common answer among the 10 human annotations."""
    counts = Counter(a["answer"] for a in answers)
    return counts.most_common(1)[0][0]

def coco_image_path(image_id: int) -> Path:
    return COCO_IMAGE_DIR / f"COCO_val2014_{image_id:012d}.jpg"

sampled_parts = []
for answer_type, group in merged.groupby("answer_type"):
    # Keep only questions whose image exists on disk
    group = group[group["image_id"].apply(
        lambda iid: coco_image_path(iid).exists()
    )].reset_index(drop=True)

    n = min(SAMPLES_PER_TYPE, len(group))
    sampled_parts.append(group.sample(n=n, random_state=42))
    print(f"  {answer_type:<10} available={len(group):,}  sampled={n}")

sample_df = pd.concat(sampled_parts, ignore_index=True).sample(
    frac=1, random_state=42).reset_index(drop=True)

all_cleaned = []
for _, row in sample_df.iterrows():
    all_cleaned.append({
        "question_id":   str(row["question_id"]),
        "image_id":      row["image_id"],
        "image_path":    str(coco_image_path(row["image_id"])),
        "question":      row["question"],
        "answer":        get_majority_answer(row["answers"]),
        "all_answers":   [a["answer"] for a in row["answers"]],
        "answer_type":   row["answer_type"],     # yes/no | number | other
        "question_type": row["question_type"],   # e.g. "what color", "how many"
    })

print(f"\nFinal sample: {len(all_cleaned)} questions")
print("Answer type breakdown:", Counter(s["answer_type"] for s in all_cleaned))


In [ ]:
# ── Helper functions ───────────────────────────────────────────────────────

# Articles/contractions to strip (same list as sst_eval_main)
_ARTICLES      = {"a", "an", "the"}
_CONTRACTIONS  = {"aint":"ain't","arent":"aren't","cant":"can't","couldve":"could've",
                   "couldnt":"couldn't","didnt":"didn't","doesnt":"doesn't","dont":"don't",
                   "hadnt":"hadn't","hasnt":"hasn't","havent":"haven't","hes":"he's",
                   "isnt":"isn't","shouldve":"should've","shouldnt":"shouldn't",
                   "theyd":"they'd","theyll":"they'll","theyre":"they're","theyve":"they've",
                   "wasnt":"wasn't","werent":"weren't","whatll":"what'll","whatre":"what're",
                   "whats":"what's","wont":"won't","wouldve":"would've","wouldnt":"wouldn't"}
_PERIOD_STRIP  = re.compile(r"(?!<=\d)(\.)(?!\d)")
_COMMA_STRIP   = re.compile(r"(\d)(,)(\d)")
_PUNCT         = set('!"&'()*+,-./:;<=?@[\\]^_`{|}~')

def normalize_answer(s: str) -> str:
    if not isinstance(s, str): s = str(s)
    s = s.lower().strip()
    s = _PERIOD_STRIP.sub("", s)
    s = _COMMA_STRIP.sub(r"\1\3", s)
    s = "".join(c for c in s if c not in _PUNCT)
    s = " ".join(w for w in s.split() if w not in _ARTICLES)
    s = " ".join(_CONTRACTIONS.get(w, w) for w in s.split())
    return s.strip()

def vqa_soft_accuracy(pred_norm: str, all_answers: list) -> float:
    """
    Official VQAv2 soft accuracy: min(num_humans_who_answered_pred / 3, 1.0).
    all_answers: raw list of 10 human answer strings.
    """
    normed = [normalize_answer(a) for a in all_answers]
    return min(normed.count(pred_norm) / 3.0, 1.0)

def count_tokens(text: str) -> int:
    if TOKEN_ENCODER:
        return len(TOKEN_ENCODER.encode(text))
    return len(text.split())

def is_lenient_correct(pred: str, gt: str) -> bool:
    return pred == gt or pred in gt or gt in pred

print("Helper functions defined.")


In [ ]:
# ── SST variant builders (identical to sst_eval_main) ─────────────────────
# NOTE: VQAv2 has no oracle scene graphs.
# All SST variants start from the *detected* SST (YOLO + colour attributes).
# The ablation tests how much information in the SST text matters.

def make_full_sst(sst: dict) -> dict:
    return {k: list(v) for k, v in sst.items()}

def make_caption_style_sst(sst: dict) -> dict:
    return {k: list(v) for k, v in sst.items()}   # same content, different prompt template

def make_no_relations_sst(sst: dict) -> dict:
    v = make_full_sst(sst); v["relations"] = []; return v

def make_no_attributes_sst(sst: dict) -> dict:
    v = make_full_sst(sst); v["attributes"] = []; return v

def make_objects_only_sst(sst: dict) -> dict:
    return {"objects": list(sst.get("objects",[])), "counts": [],
            "attributes": [], "relations": [], "text": []}

def sst_to_prompt_structured(sst: dict) -> str:
    lines = []
    if sst.get("objects"):
        lines.append("Objects: " + ", ".join(sst["objects"][:15]))
    if sst.get("attributes"):
        lines.append("Attributes: " + "; ".join(sst["attributes"][:15]))
    if sst.get("relations"):
        lines.append("Relations: " + "; ".join(sst["relations"][:10]))
    if sst.get("counts"):
        lines.append("Counts: " + ", ".join(sst["counts"][:8]))
    if sst.get("text"):
        lines.append("Visible text: " + ", ".join(sst["text"][:5]))
    return "\n".join(lines) if lines else "No scene information available."

# Keyword extraction for keyword-aware variants
_STOPWORDS = {"the","a","an","is","are","was","were","in","on","at","of","and",
              "or","to","it","its","this","that","there","do","does","did",
              "what","which","where","when","who","how","does","with","for",
              "be","been","have","has","had","not","no","any","some","can",
              "could","would","should","will","would","each","many","much",
              "color","colour","type","kind","object","thing","picture","image"}

def extract_keywords(question: str) -> set:
    return {w for w in re.findall(r"[a-z]+", question.lower()) if w not in _STOPWORDS and len(w) > 2}

def filter_sst_by_keywords(sst: dict, keywords: set) -> dict:
    if not keywords:
        return make_full_sst(sst)
    def keep(item):
        return any(kw in item.lower() for kw in keywords)
    return {
        "objects":    [o for o in sst.get("objects",[])    if keep(o)]  or list(sst.get("objects",[])),
        "attributes": [a for a in sst.get("attributes",[]) if keep(a)]  or list(sst.get("attributes",[])),
        "relations":  [r for r in sst.get("relations",[])  if keep(r)]  or list(sst.get("relations",[])),
        "counts":     list(sst.get("counts",[])),
        "text":       list(sst.get("text",[])),
    }

def make_compact_keyword_sst(sst: dict, question: str) -> dict:
    keywords = extract_keywords(question)
    v = filter_sst_by_keywords(sst, keywords)
    # Keep only top-5 per field to reduce tokens
    return {k: v[k][:5] for k in v}

print("SST variant builders defined.")


In [ ]:
# ── build_method_prompt ────────────────────────────────────────────────────

def build_standard_prompt(question: str, scene_prompt: str) -> str:
    return (
        "You are answering a visual question based on a structured scene description.\n"
        "Scene description:\n"
        f"{scene_prompt}\n\n"
        f"Question: {question}\n\n"
        "Rules:\n"
        "- Answer with a single word or very short phrase.\n"
        "- Do not explain your reasoning.\n"
        "- If you are not sure, give your best guess.\n\n"
        "Final answer:"
    )

def build_caption_prompt(question: str, scene_prompt: str) -> str:
    return (
        "The following is a caption-style scene description:\n"
        f"{scene_prompt}\n\n"
        f"Based on this description, answer: {question}\n"
        "Answer (one word or short phrase):"
    )

def build_method_prompt(sample: dict, sst: dict, method: str):
    """Returns (full_prompt, scene_prompt_only)."""
    q = sample["question"]
    if method == "full_sst":
        v = make_full_sst(sst);       sp = sst_to_prompt_structured(v); p = build_standard_prompt(q, sp)
    elif method == "caption_style_sst":
        v = make_caption_style_sst(sst); sp = sst_to_prompt_structured(v); p = build_caption_prompt(q, sp)
    elif method == "keyword_aware_sst":
        kw = extract_keywords(q);     v  = filter_sst_by_keywords(sst, kw)
        sp = sst_to_prompt_structured(v); p = build_standard_prompt(q, sp)
    elif method == "compact_keyword_sst":
        v  = make_compact_keyword_sst(sst, q)
        sp = sst_to_prompt_structured(v); p = build_standard_prompt(q, sp)
    elif method == "no_relations_sst":
        v = make_no_relations_sst(sst); sp = sst_to_prompt_structured(v); p = build_standard_prompt(q, sp)
    elif method == "no_attributes_sst":
        v = make_no_attributes_sst(sst); sp = sst_to_prompt_structured(v); p = build_standard_prompt(q, sp)
    elif method == "objects_only_sst":
        v = make_objects_only_sst(sst); sp = sst_to_prompt_structured(v); p = build_standard_prompt(q, sp)
    elif method == "question_only":
        sp = "no scene information"
        _q = sample["question"]
        p  = (
            "Answer the following visual question as best you can.\n"
            "You do not have access to the image.\n\n"
            f"Question: {_q}\n\n"
            "Rules:\n"
            "- Give only the final answer.\n"
            "- Do not explain.\n"
            "- If you cannot answer, say unknown.\n\n"
            "Final answer:"
        )
    else:
        raise ValueError(f"Unknown method: {method}")
    return p, sp

METHODS = [
    "full_sst",
    "caption_style_sst",
    "keyword_aware_sst",
    "compact_keyword_sst",
    "no_relations_sst",
    "no_attributes_sst",
    "objects_only_sst",
    "question_only",   # language-prior baseline
]

METHOD_NAMES = {
    "full_sst":           "Full SST",
    "caption_style_sst":  "Caption-Style SST",
    "keyword_aware_sst":  "Keyword-Aware SST",
    "compact_keyword_sst":"Compact Keyword SST",
    "no_relations_sst":   "No-Relations SST",
    "no_attributes_sst":  "No-Attributes SST",
    "objects_only_sst":   "Objects-Only SST",
    "question_only":      "Question-Only (no scene)",
}

print("build_method_prompt and METHODS defined.")
print("Methods:", METHODS)


In [ ]:
# ── image_to_detected_sst ──────────────────────────────────────────────────
# Identical to sst_eval_main v2: YOLO detection + colour/size attributes + EasyOCR

_yolo_model = None
def get_yolo():
    global _yolo_model
    if _yolo_model is None:
        assert YOLO is not None, "ultralytics not installed"
        _yolo_model = YOLO("yolov8n.pt")
    return _yolo_model

def boxes_to_relations(detections: list, iou_thresh: float = 0.0) -> list:
    relations = []
    for i, a in enumerate(detections):
        for j, b in enumerate(detections):
            if i >= j: continue
        ax1, ay1, ax2, ay2 = a["bbox"]
        bx1, by1, bx2, by2 = b["bbox"]
        acx, acy = (ax1+ax2)/2, (ay1+ay2)/2
        bcx, bcy = (bx1+bx2)/2, (by1+by2)/2
        dx, dy = bcx-acx, bcy-acy
        if abs(dx) > abs(dy):
            rel = "to the right of" if dx > 0 else "to the left of"
        else:
            rel = "below" if dy > 0 else "above"
        relations.append(f"{a['label']} {rel} {b['label']}")
    return relations[:15]

def image_to_detected_sst(image_path, conf_thresh: float = 0.35) -> dict:
    sst = {"objects": [], "counts": [], "attributes": [], "relations": [], "text": []}
    image_path = Path(image_path)
    if not image_path.exists():
        return sst

    # ── Step 1: YOLO detection ───────────────────────────────────────────────
    model = get_yolo()
    results = model(str(image_path), conf=conf_thresh, verbose=False)
    detections = []
    for box in results[0].boxes:
        label = model.names[int(box.cls[0])]
        conf  = float(box.conf[0])
        bbox  = box.xyxy[0].tolist()
        detections.append({"label": label, "conf": conf, "bbox": bbox})

    # ── Step 2: Objects + counts ─────────────────────────────────────────────
    sst["objects"] = list(dict.fromkeys(d["label"] for d in detections))[:15]
    counts = Counter(d["label"] for d in detections)
    sst["counts"] = [f"{cnt} {lbl}" for lbl, cnt in counts.most_common(8) if cnt > 1]

    # ── Step 3: Colour + size attributes ────────────────────────────────────
    try:
        import numpy as _np
        img_bgr = cv2.imread(str(image_path))
        if img_bgr is not None:
            ih, iw = img_bgr.shape[:2]
            img_area = ih * iw
            for det in detections:
                x1,y1,x2,y2 = [int(v) for v in det["bbox"]]
                x1,y1 = max(0,x1), max(0,y1)
                x2,y2 = min(iw,x2), min(ih,y2)
                crop = img_bgr[y1:y2, x1:x2]
                if crop.size == 0: continue
                lbl = det["label"]

                # Dominant colour via mean HSV
                crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                pixels = crop_rgb.reshape(-1,3).astype(_np.float32)
                if len(pixels) > 500:
                    pixels = pixels[_np.random.choice(len(pixels),500,replace=False)]
                mean_rgb = pixels.mean(axis=0)
                hsv = cv2.cvtColor(_np.uint8([[mean_rgb.astype(_np.uint8)]]),
                                   cv2.COLOR_RGB2HSV)[0][0]
                h,s,v = int(hsv[0]),int(hsv[1]),int(hsv[2])
                if v < 50:               colour = "black"
                elif v > 200 and s < 40: colour = "white"
                elif s < 40:             colour = "gray"
                elif h < 10 or h > 160:  colour = "red"
                elif h < 25:             colour = "orange"
                elif h < 35:             colour = "yellow"
                elif h < 85:             colour = "green"
                elif h < 130:            colour = "blue"
                else:                    colour = "purple"
                sst["attributes"].append(f"{lbl}: {colour}")

                # Relative size
                frac = ((x2-x1)*(y2-y1)) / img_area if img_area > 0 else 0
                size = "large" if frac > 0.25 else ("medium" if frac > 0.06 else "small")
                sst["attributes"].append(f"{lbl}: {size}")

            seen = set(); deduped = []
            for a in sst["attributes"]:
                if a not in seen: deduped.append(a); seen.add(a)
            sst["attributes"] = deduped[:20]
    except Exception:
        pass

    # ── Step 4: Spatial relations ────────────────────────────────────────────
    sst["relations"] = boxes_to_relations(detections)

    # ── Step 5: EasyOCR text ─────────────────────────────────────────────────
    if OCR_READER is not None:
        try:
            ocr_results = OCR_READER.readtext(str(image_path), detail=0)
            sst["text"] = [t.strip() for t in ocr_results if len(t.strip()) > 1][:8]
        except Exception:
            pass

    return sst

print("image_to_detected_sst defined.")


In [ ]:
# ── Inference functions ────────────────────────────────────────────────────

def ollama_query(prompt: str, model: str = "mistral", timeout: int = 120):
    """Query Mistral via ollama.chat. Returns (response_text, elapsed_seconds)."""
    assert ollama is not None, "ollama package is not installed. Run: pip install ollama"
    t0 = time.perf_counter()
    resp = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "num_predict": 20},
    )
    return resp["message"]["content"].strip(), time.perf_counter() - t0

def encode_b64(path) -> str:
    return base64.b64encode(Path(path).read_bytes()).decode()

def query_llava(img_path, question: str):
    """Query LLaVA via ollama.chat (same interface as Mistral for fair latency comparison)."""
    assert ollama is not None, "ollama package is not installed. Run: pip install ollama"
    prompt = (
        "Answer the question based only on the image.\n"
        "Be concise. Answer with a short phrase or single word.\n"
        f"Question: {question}\nAnswer:"
    )
    img_b64 = encode_b64(img_path)
    t0 = time.perf_counter()
    resp = ollama.chat(
        model="llava",
        messages=[{"role": "user", "content": prompt, "images": [img_b64]}],
        options={"temperature": 0, "num_predict": 20},
    )
    return resp["message"]["content"].strip(), time.perf_counter() - t0

print("ollama_query and query_llava defined.")


In [ ]:
# ── Build detected SSTs for all sample images ─────────────────────────────
# Saves to data/vqav2/detected_ssts.json — skips if already exists.

SST_CACHE = VQAV2_DIR / "detected_ssts.json"

if SST_CACHE.exists():
    with open(SST_CACHE) as f:
        detected_ssts = json.load(f)
    print(f"Loaded SST cache: {len(detected_ssts)} images")
else:
    detected_ssts = {}

unique_images = list(dict.fromkeys(s["image_path"] for s in all_cleaned))
remaining     = [p for p in unique_images if p not in detected_ssts]
print(f"Total unique images: {len(unique_images)}  |  Already cached: {len(detected_ssts)}  |  To process: {len(remaining)}")

for idx, img_path in enumerate(remaining):
    detected_ssts[img_path] = image_to_detected_sst(img_path)
    if (idx + 1) % 50 == 0:
        with open(SST_CACHE, "w") as f:
            json.dump(detected_ssts, f)
        print(f"  {idx+1}/{len(remaining)} done, cache saved")

with open(SST_CACHE, "w") as f:
    json.dump(detected_ssts, f)
print(f"Done. {len(detected_ssts)} SSTs cached → {SST_CACHE}")


In [ ]:
# ── Main evaluation loop ───────────────────────────────────────────────────

# Load checkpoint
if CHECKPOINT.exists():
    ckpt_df  = pd.read_csv(CHECKPOINT, low_memory=False)
    done_set = set(zip(ckpt_df["question_id"].astype(str), ckpt_df["pipeline"]))
    print(f"Resuming from checkpoint: {len(ckpt_df):,} rows done")
else:
    ckpt_df  = pd.DataFrame()
    done_set = set()
    print("No checkpoint found — starting fresh")

all_rows = []
total_calls     = len(all_cleaned) * len(METHODS)
completed_calls = len(done_set)
print(f"Total calls: {total_calls:,} | Completed: {completed_calls:,} | Remaining: {total_calls - completed_calls:,}")

for i, sample in enumerate(all_cleaned):
    qid      = str(sample["question_id"])
    img_path = sample["image_path"]
    sst      = detected_ssts.get(img_path, {
        "objects":[], "counts":[], "attributes":[], "relations":[], "text":[]
    })
    new_rows = []

    for method in METHODS:
        if (qid, method) in done_set:
            continue
        try:
            prompt, scene_prompt = build_method_prompt(sample, sst, method)
            raw_pred, elapsed    = ollama_query(prompt)
            pred_norm            = normalize_answer(raw_pred)
            gt_norm              = normalize_answer(sample["answer"])

            exact   = int(pred_norm == gt_norm)
            lenient = int(is_lenient_correct(pred_norm, gt_norm))
            soft    = vqa_soft_accuracy(pred_norm, sample["all_answers"])
            tokens  = count_tokens(scene_prompt)

            new_rows.append({
                "question_id":   qid,
                "image_id":      sample["image_id"],
                "question":      sample["question"],
                "true_answer":   sample["answer"],
                "all_answers":   "|".join(sample["all_answers"]),
                "answer_type":   sample["answer_type"],
                "question_type": sample["question_type"],
                "pipeline":      method,
                "prediction":    raw_pred,
                "prediction_norm": pred_norm,
                "exact_correct": exact,
                "lenient_correct": lenient,
                "soft_accuracy": soft,
                "scene_tokens":  tokens,
                "latency_s":     round(elapsed, 3),
            })
        except Exception as e:
            print(f"  ⚠️  Sample {qid} ({method}) failed: {e}")

    all_rows.extend(new_rows)
    done_set.update((qid, r["pipeline"]) for r in new_rows)

    if new_rows and (i + 1) % CHECKPOINT_FREQ == 0:
        new_df   = pd.DataFrame(new_rows if not ckpt_df.empty else all_rows)
        ckpt_df  = pd.concat([ckpt_df, pd.DataFrame(all_rows)], ignore_index=True).drop_duplicates(
            subset=["question_id","pipeline"], keep="last")
        ckpt_df.to_csv(CHECKPOINT, index=False)
        all_rows = []
        pct = len(done_set) / total_calls * 100
        print(f"  {i+1}/{len(all_cleaned)} questions | {len(done_set):,}/{total_calls:,} calls ({pct:.1f}%)")

# Final save
if all_rows:
    ckpt_df = pd.concat([ckpt_df, pd.DataFrame(all_rows)], ignore_index=True).drop_duplicates(
        subset=["question_id","pipeline"], keep="last")
    ckpt_df.to_csv(CHECKPOINT, index=False)

print(f"\nEvaluation complete. {len(ckpt_df):,} rows saved → {CHECKPOINT}")


In [ ]:
# ── Summarize results ──────────────────────────────────────────────────────
results_df = pd.read_csv(CHECKPOINT, low_memory=False)
print(f"Loaded {len(results_df):,} rows")

summary_rows = []
for method in METHODS:
    sub = results_df[results_df["pipeline"] == method]
    if sub.empty: continue
    summary_rows.append({
        "method":          method,
        "method_name":     METHOD_NAMES[method],
        "n":               len(sub),
        "exact_accuracy":  sub["exact_correct"].mean(),
        "lenient_accuracy":sub["lenient_correct"].mean(),
        "soft_accuracy":   sub["soft_accuracy"].mean(),   # VQAv2 official metric
        "avg_tokens":      sub["scene_tokens"].mean(),
        "avg_latency_ms":  sub["latency_s"].mean() * 1000,
    })

summary_df = pd.DataFrame(summary_rows)
# Token reduction relative to full_sst
full_tok = summary_df.loc[summary_df["method"]=="full_sst","avg_tokens"].values[0]
summary_df["token_reduction_pct"] = ((full_tok - summary_df["avg_tokens"]) / full_tok * 100).round(1)
summary_df["soft_accuracy_pct"]   = (summary_df["soft_accuracy"] * 100).round(2)
summary_df["exact_accuracy_pct"]  = (summary_df["exact_accuracy"] * 100).round(2)

print("\n── VQAv2 Summary ─────────────────────────────────────────────────────────")
print(summary_df[["method_name","exact_accuracy_pct","soft_accuracy_pct",
                   "avg_tokens","token_reduction_pct","avg_latency_ms"]].to_string(index=False))

summary_df.to_csv(RESULTS_DIR / f"{PREFIX}_summary.csv", index=False)
print(f"\nSummary saved → {RESULTS_DIR}/{PREFIX}_summary.csv")


In [ ]:
# ── LLaVA evaluation on VQAv2 ─────────────────────────────────────────────
# Evaluates on the same questions as the SST pipeline.
# Uses ollama.chat for fair latency comparison.

LLAVA_N   = min(500, len(all_cleaned))   # cap at 500 — LLaVA is slow
llava_rows = []

if LLAVA_OUT_V2.exists():
    llava_df = pd.read_csv(LLAVA_OUT_V2, low_memory=False)
    llava_done = set(llava_df["question_id"].astype(str))
    print(f"LLaVA checkpoint: {len(llava_df)} rows")
else:
    llava_df   = pd.DataFrame()
    llava_done = set()

subset = all_cleaned[:LLAVA_N]
print(f"Running LLaVA on {len([s for s in subset if str(s['question_id']) not in llava_done])} remaining samples...")

for i, sample in enumerate(subset):
    qid = str(sample["question_id"])
    if qid in llava_done:
        continue
    try:
        raw_pred, elapsed = query_llava(sample["image_path"], sample["question"])
        pred_norm = normalize_answer(raw_pred)
        gt_norm   = normalize_answer(sample["answer"])
        llava_rows.append({
            "question_id":  qid,
            "image_id":     sample["image_id"],
            "question":     sample["question"],
            "ground_truth": sample["answer"],
            "all_answers":  "|".join(sample["all_answers"]),
            "answer_type":  sample["answer_type"],
            "prediction":   raw_pred,
            "pred_norm":    pred_norm,
            "gt_norm":      gt_norm,
            "exact":        int(pred_norm == gt_norm),
            "soft_accuracy":vqa_soft_accuracy(pred_norm, sample["all_answers"]),
            "latency_s":    round(elapsed, 3),
        })
    except Exception as e:
        print(f"  ⚠️  {qid} failed: {e}")

    if llava_rows and (i + 1) % 50 == 0:
        llava_df = pd.concat([llava_df, pd.DataFrame(llava_rows)], ignore_index=True)
        llava_df.to_csv(LLAVA_OUT_V2, index=False)
        llava_rows = []
        print(f"  {i+1}/{LLAVA_N} done")

if llava_rows:
    llava_df = pd.concat([llava_df, pd.DataFrame(llava_rows)], ignore_index=True)
    llava_df.to_csv(LLAVA_OUT_V2, index=False)

print(f"\nLLaVA evaluation done. {len(llava_df)} rows → {LLAVA_OUT_V2}")


In [ ]:
# ── LLaVA summary ─────────────────────────────────────────────────────────
if LLAVA_OUT_V2.exists():
    llava_df = pd.read_csv(LLAVA_OUT_V2, low_memory=False)
    llava_summary_v2 = {
        "exact_accuracy":   llava_df["exact"].mean(),
        "soft_accuracy":    llava_df["soft_accuracy"].mean(),
        "avg_latency_sec":  llava_df["latency_s"].mean(),
        "n":                len(llava_df),
    }
    print(f"LLaVA exact accuracy:  {llava_summary_v2['exact_accuracy']:.1%}")
    print(f"LLaVA soft accuracy:   {llava_summary_v2['soft_accuracy']:.1%}")
    print(f"LLaVA avg latency:     {llava_summary_v2['avg_latency_sec']*1000:.0f} ms")
    print(f"LLaVA n:               {llava_summary_v2['n']}")
else:
    llava_summary_v2 = None
    print("LLaVA results not available — run the LLaVA evaluation cell first.")


In [ ]:
# ── Results plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("VQAv2 Results — SST Methods", fontsize=13, y=1.02)

_palette   = ["#2C3E50","#3498DB","#27AE60","#16A085","#E74C3C","#E67E22","#95A5A6","#BDC3C7","#8E44AD","#1ABC9C"]
method_names_plot = summary_df["method_name"].tolist()
bar_colors = [_palette[i % len(_palette)] for i in range(len(method_names_plot))]

# ── Plot 1: Soft accuracy (VQAv2 official) ────────────────────────────────
ax = axes[0]
soft_accs = (summary_df["soft_accuracy"] * 100).tolist()
ax.bar(range(len(method_names_plot)), soft_accs, color=bar_colors, edgecolor="white")
if llava_summary_v2:
    llava_soft = llava_summary_v2["soft_accuracy"] * 100
    ax.axhline(llava_soft, color="red", linestyle="--", linewidth=1.8,
               label=f"LLaVA ({llava_soft:.1f}%)")
    ax.legend(fontsize=9)
ax.set_xticks(range(len(method_names_plot)))
ax.set_xticklabels([m.replace(" SST","") for m in method_names_plot], rotation=35, ha="right", fontsize=8)
ax.set_ylabel("Soft accuracy (%)", fontsize=10)
ax.set_title("VQAv2 soft accuracy per method", fontsize=11)
ax.grid(axis="y", alpha=0.3)

# ── Plot 2: Soft accuracy by answer type ─────────────────────────────────
ax2 = axes[1]
best_method = summary_df.loc[summary_df["soft_accuracy"].idxmax(), "method"]
best_sub    = results_df[results_df["pipeline"] == best_method]
type_acc    = best_sub.groupby("answer_type")["soft_accuracy"].mean() * 100
type_acc.plot(kind="bar", ax=ax2, color=["#3498DB","#27AE60","#E74C3C"], edgecolor="white")
ax2.set_title(f"Soft acc by answer type\n(best method: {METHOD_NAMES[best_method]})", fontsize=11)
ax2.set_xlabel(""); ax2.set_ylabel("Soft accuracy (%)", fontsize=10)
ax2.tick_params(axis="x", rotation=0)
ax2.grid(axis="y", alpha=0.3)

# ── Plot 3: Token count vs soft accuracy ─────────────────────────────────
ax3 = axes[2]
for i, row in summary_df.iterrows():
    ax3.scatter(row["avg_tokens"], row["soft_accuracy"]*100,
                color=bar_colors[i], s=80, zorder=3)
    ax3.annotate(row["method_name"].replace(" SST",""),
                 (row["avg_tokens"], row["soft_accuracy"]*100),
                 fontsize=7, textcoords="offset points", xytext=(5,2))
ax3.set_xlabel("Avg scene tokens", fontsize=10)
ax3.set_ylabel("Soft accuracy (%)", fontsize=10)
ax3.set_title("Tokens vs accuracy (efficiency frontier)", fontsize=11)
ax3.grid(alpha=0.3)

plt.tight_layout()
fig_path = RESULTS_DIR / f"{PREFIX}_results.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved → {fig_path}")


In [ ]:
# ── Cross-benchmark comparison: VQAv2 vs GQA ──────────────────────────────
# Loads GQA summary from sst_eval_main results and compares method-by-method.
# Run sst_eval_main.ipynb first so the GQA summary CSV exists.

import glob
gqa_summary_files = sorted(glob.glob(str(GQA_RESULTS / "gqa_*_sst_eval_summary.csv")))

if not gqa_summary_files:
    print("GQA summary not found — run sst_eval_main.ipynb first.")
else:
    gqa_summary = pd.read_csv(gqa_summary_files[-1])   # use the most recent run
    print(f"GQA summary loaded: {gqa_summary_files[-1]}")

    # Align on method names
    gqa_summary["benchmark"] = "GQA (oracle SST)"
    summary_df["benchmark"]  = "VQAv2 (detected SST)"

    combined = pd.concat([
        gqa_summary[["method_name","exact_accuracy","benchmark"]],
        summary_df[["method_name","exact_accuracy","benchmark"]],
    ], ignore_index=True)

    pivot = combined.pivot(index="method_name", columns="benchmark", values="exact_accuracy") * 100
    pivot = pivot.round(2).sort_values("GQA (oracle SST)", ascending=False)

    print("\n── Cross-benchmark exact accuracy (%) ─────────────────────────────────")
    print(pivot.to_string())

    # Plot side-by-side bars
    fig, ax = plt.subplots(figsize=(12, 5))
    x = range(len(pivot))
    w = 0.35
    bars1 = ax.bar([i - w/2 for i in x], pivot.get("GQA (oracle SST)", [0]*len(pivot)),
                   width=w, label="GQA (oracle SST)",    color="#3498DB", edgecolor="white")
    bars2 = ax.bar([i + w/2 for i in x], pivot.get("VQAv2 (detected SST)", [0]*len(pivot)),
                   width=w, label="VQAv2 (detected SST)", color="#E74C3C", edgecolor="white")
    ax.set_xticks(list(x))
    ax.set_xticklabels(pivot.index, rotation=30, ha="right", fontsize=9)
    ax.set_ylabel("Exact accuracy (%)", fontsize=11)
    ax.set_title("GQA vs VQAv2 — SST method comparison", fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    fig_path = RESULTS_DIR / "cross_benchmark_comparison.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figure saved → {fig_path}")

    pivot.to_csv(RESULTS_DIR / "cross_benchmark_comparison.csv")
    print(f"Table saved → {RESULTS_DIR}/cross_benchmark_comparison.csv")
